# Tiki Electronics – Data Quality & Preprocessing Pipeline


| Section | Nội dung |
|---------|----------|
| **1** | Load & Audit (tổng quan dữ liệu) |
| **2** | Drop Dead Columns (cột toàn 0) |
| **3** | Phân tầng `purchase_status` (has_sales / new_listing) |
| **4** | Rating Anomaly – Suspect Rate theo brand_type |
| **5** | Discount Flag – Phát hiện giảm giá ảo |
| **6** | Assertions – Kiểm tra tính toàn vẹn |
| **7** | Export processed CSV + preprocessing report |

---
> **Không drop rows** – Chỉ thêm columns mới, không xóa bất kỳ dòng nào để bảo toàn toàn bộ dữ liệu gốc.

## Configuration

In [1]:
import json
import os

# ── Đọc path từ config.json của crawler (không hardcode) ──────────────────────
CONFIG_PATH = "../config.json"   # Relative từ thư mục preprocessing/

with open(CONFIG_PATH, encoding="utf-8") as f:
    _cfg = json.load(f)

# Đường dẫn gốc của project
PROJECT_ROOT = os.path.dirname(os.path.abspath(CONFIG_PATH))

# Input CSV: file clean từ crawler
INPUT_CSV  = os.path.join(PROJECT_ROOT, _cfg["output"]["csv_export"])

# Output CSV: file đã qua preprocessing
OUTPUT_CSV = INPUT_CSV.replace(".csv", "_processed.csv")
OUTPUT_DB = OUTPUT_CSV.replace(".csv", ".db")

# Output report JSON
REPORT_JSON = os.path.join(PROJECT_ROOT, "data", "preprocessing_report.json")

# ── Tham số phân loại (có thể thay đổi) ──────────────────────────────────────
SUSPECT_RATING_THRESHOLD   = 4.5   # rating_average > ngưỡng này
SUSPECT_REVIEW_MIN         = 1     # review_count >= min (loại bỏ sp chưa có review)
SUSPECT_REVIEW_MAX         = 10    # review_count < max (quá ít review)
EXTREME_DISCOUNT_THRESHOLD = 50    # discount_rate >= % này → extreme
DEAD_COLUMN_THRESHOLD      = 0.95  # Cột có > 95% giá trị = 0 → drop

print(f"📂 Input  CSV : {INPUT_CSV}")
print(f"📂 Output CSV : {OUTPUT_CSV}")
print(f"📄 Report JSON: {REPORT_JSON}")

📂 Input  CSV : c:\Users\Admin\Documents\GitHub\Ecom-Data-Crawler\data/tiki_electronics_2026.csv
📂 Output CSV : c:\Users\Admin\Documents\GitHub\Ecom-Data-Crawler\data/tiki_electronics_2026_processed.csv
📄 Report JSON: c:\Users\Admin\Documents\GitHub\Ecom-Data-Crawler\data\preprocessing_report.json


In [9]:
%pip install pandas numpy

  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.4 MB 2.8 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/12.4 MB 3.8 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.4 MB 4.4 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.4 MB 4.4 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.4 MB 4.4 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.4 MB 4.4 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.4 MB 4.4 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.4 MB 1.5 MB/s eta 0:00:07
   ------------ --------------------------- 

---
## Section 1 – Load & Audit

In [2]:
import pandas as pd
import numpy as np

# Load dữ liệu
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
ORIGINAL_ROW_COUNT = len(df)  # Lưu lại để kiểm tra sau

print(f"{'='*60}")
print(f"TỔNG QUAN DỮ LIỆU – {os.path.basename(INPUT_CSV)}")
print(f"{'='*60}")
print(f"  Số dòng (sản phẩm) : {len(df):,}")
print(f"  Số cột             : {len(df.columns)}")
print(f"  Kích thước bộ nhớ  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()

# ── Kiểm tra giá trị thiếu ────────────────────────────────────────────────────
print("Giá trị thiếu (null/NaN):")
null_info = df.isnull().sum()
null_pct  = (null_info / len(df) * 100).round(2)
null_df = pd.DataFrame({"null_count": null_info, "null_pct": null_pct})
null_df = null_df[null_df["null_count"] > 0]
print(null_df.to_string() if not null_df.empty else "  ✅ Không có giá trị null")
print()

# ── Kiểm tra trùng lặp ────────────────────────────────────────────────────────
dup_count = df.duplicated(subset=["product_id"]).sum()
print(f"Sản phẩm trùng product_id: {dup_count} {'⚠️' if dup_count > 0 else '✅ Không có'}")
print()

# ── Tổng quan phân bố brand_type ─────────────────────────────────────────────
print("Phân bố brand_type:")
bt_dist = df["brand_type"].value_counts()
for bt, cnt in bt_dist.items():
    print(f"  {bt:20s}: {cnt:>6,} ({cnt/len(df)*100:.1f}%)")

df.head(3)

TỔNG QUAN DỮ LIỆU – tiki_electronics_2026.csv
  Số dòng (sản phẩm) : 10,978
  Số cột             : 16
  Kích thước bộ nhớ  : 7.90 MB

Giá trị thiếu (null/NaN):
            null_count  null_pct
brand_name           1      0.01

Sản phẩm trùng product_id: 0 ✅ Không có

Phân bố brand_type:
  Local_Generic       :  6,307 (57.5%)
  Global_Brand        :  4,514 (41.1%)
  OEM_Generic         :    157 (1.4%)


,product_id,product_name,category_id,category_name,brand_name,brand_type,price,original_price,discount_rate,rating_average,review_count,quantity_sold,seller_name,is_tiki_trading,first_crawled_at,last_crawled_at
0,274743461,Tai Nghe Không Dây Chống Ồn Chủ Động ANC Quiet...,1804,Tai Nghe Có Dây,Audio-technica,Global_Brand,7990000,7990000,0.0,0.0,0,0,AUDIO TECHNICA OFFICIAL STORE,0,2026-03-11 11:48:56,2026-03-11 11:48:56
1,274695799,Tai Nghe Chụp Tai Audio Technica ATH-M50XSTS-U...,1804,Tai Nghe Có Dây,Audio-technica,Global_Brand,7090000,7090000,0.0,0.0,0,0,AUDIO TECHNICA OFFICIAL STORE,0,2026-03-11 11:48:56,2026-03-11 11:48:56
2,274754463,Tai Nghe Kiểm Âm Choàng Đầu Audio Technica ATH...,1804,Tai Nghe Có Dây,Audio-technica,Global_Brand,6900000,6900000,0.0,0.0,0,0,AUDIO TECHNICA OFFICIAL STORE,0,2026-03-11 11:48:56,2026-03-11 11:48:56


---
## Section 2 – Drop Dead Columns

> **Chiến lược tổng quát:** Bất kỳ cột numeric nào có > 95% giá trị = 0 sẽ được đánh dấu và loại bỏ.
> Điều này xử lý `is_official` trong dữ liệu cũ, và bảo vệ cho các lần crawl sau.

In [3]:
print("SỬ LÝ DEAD COLUMNS (cột có > {:.0%} giá trị = 0)".format(DEAD_COLUMN_THRESHOLD))
print("-" * 60)

# Chỉ kiểm tra cột numeric
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

dead_columns = []
for col in numeric_cols:
    zero_pct = (df[col] == 0).mean()
    if zero_pct > DEAD_COLUMN_THRESHOLD:
        dead_columns.append(col)
        print(f"  ⚠️  '{col}': {zero_pct:.1%} giá trị = 0 → SẼ BỊ DROP")

if not dead_columns:
    print("  ✅ Không tìm thấy dead column nào.")
else:
    df.drop(columns=dead_columns, inplace=True)
    print(f"\n  → Đã drop {len(dead_columns)} cột: {dead_columns}")
    print(f"  → Số cột còn lại: {len(df.columns)}")

print(f"\n✅ Các cột hiện tại: {list(df.columns)}")

SỬ LÝ DEAD COLUMNS (cột có > 95% giá trị = 0)
------------------------------------------------------------
  ✅ Không tìm thấy dead column nào.

✅ Các cột hiện tại: ['product_id', 'product_name', 'category_id', 'category_name', 'brand_name', 'brand_type', 'price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'quantity_sold', 'seller_name', 'is_tiki_trading', 'first_crawled_at', 'last_crawled_at']


---
## Section 3 – Phân tầng `purchase_status`

Phân chia sản phẩm thành 2 nhóm phục vụ phân tích insight:
- **`has_sales`**: Đã có hoạt động (quantity_sold > 0 OR review_count > 0)
- **`new_listing`**: Chưa bán được / mới đăng (quantity_sold = 0 AND review_count = 0)

> Lưu ý: Nhóm `new_listing` bị loại ra khỏi phân tích rating/review để tránh bias.

In [4]:
# ── Tạo cột purchase_status ───────────────────────────────────────────────────
def assign_purchase_status(row):
    """
    Phân tầng sản phẩm thành has_sales / new_listing.
    Logic: nếu có BẤT KỲ hoạt động nào (bán hoặc review) → has_sales.
    """
    if (row["quantity_sold"] > 0) or (row["review_count"] > 0):
        return "has_sales"
    return "new_listing"

df["purchase_status"] = df.apply(assign_purchase_status, axis=1)

# ── Thống kê ──────────────────────────────────────────────────────────────────
print("PHÂN TẦNG PURCHASE STATUS")
print("-" * 60)
status_counts = df["purchase_status"].value_counts()
for status, cnt in status_counts.items():
    pct = cnt / len(df) * 100
    print(f"  {status:15s}: {cnt:>6,} ({pct:.1f}%)")
print()

# ── Phân bố purchase_status theo brand_type ───────────────────────────────────
print("Phân bố theo brand_type:")
cross_tab = pd.crosstab(
    df["brand_type"],
    df["purchase_status"],
    margins=True,
    margins_name="TOTAL"
)
# Thêm % new_listing
cross_tab["new_listing_%"] = (
    cross_tab.get("new_listing", 0) / cross_tab["TOTAL"] * 100
).round(1)
print(cross_tab.to_string())
print()
print("✅ Cột 'purchase_status' đã được thêm vào DataFrame.")

PHÂN TẦNG PURCHASE STATUS
------------------------------------------------------------
  new_listing    :  5,496 (50.1%)
  has_sales      :  5,482 (49.9%)

Phân bố theo brand_type:
purchase_status  has_sales  new_listing  TOTAL  new_listing_%
brand_type                                                   
Global_Brand          2397         2117   4514           46.9
Local_Generic         2970         3337   6307           52.9
OEM_Generic            115           42    157           26.8
TOTAL                 5482         5496  10978           50.1

✅ Cột 'purchase_status' đã được thêm vào DataFrame.


---
## Section 4 – Rating Anomaly: Suspect Rate

**Định nghĩa sản phẩm bất thường (suspect):**
- `rating_average > 4.5` (rating rất cao)
- `review_count >= 1` (có review thực, không phải chưa ai mua)
- `review_count < 10` (quá ít review để tin cậy)

> **Cơ sở lý luận:** Sản phẩm có rating 5.0 nhưng chỉ 2 review là dấu hiệu review không đại diện hoặc giả tạo.

In [7]:
print("RATING ANOMALY – SUSPECT RATE ANALYSIS")
print("-" * 60)
print(f"  Ngưỡng: rating > {SUSPECT_RATING_THRESHOLD} AND {SUSPECT_REVIEW_MIN} <= review_count < {SUSPECT_REVIEW_MAX}")
print()

# ── Lọc tập sản phẩm đủ điều kiện đánh giá (có review thực) ──────────────────
df_with_reviews = df[
    (df["review_count"] >= SUSPECT_REVIEW_MIN) &
    (df["purchase_status"] == "has_sales")
].copy()

print(f"Tổng sản phẩm có review thực (review >= {SUSPECT_REVIEW_MIN}): {len(df_with_reviews):,}")
print(f"  (loại bỏ {len(df) - len(df_with_reviews):,} sản phẩm new_listing / 0 review)")
print()

# ── Đánh dấu suspect ─────────────────────────────────────────────────────────
def is_rating_suspect(row):
    """
    True nếu sản phẩm có dấu hiệu rating bất thường:
    Rating rất cao nhưng số lượng review quá ít để tin cậy.
    """
    return (
        (row["rating_average"] > SUSPECT_RATING_THRESHOLD) and
        (SUSPECT_REVIEW_MIN <= row["review_count"] < SUSPECT_REVIEW_MAX)
    )

df["is_rating_suspect"] = df.apply(is_rating_suspect, axis=1)

# ── Thống kê theo brand_type ──────────────────────────────────────────────────
print(f"{'brand_type':<20} {'eligible':>10} {'suspect':>10} {'suspect_rate':>14}")
print("-" * 60)

summary_rows = []
for bt in sorted(df["brand_type"].unique()):
    eligible = df_with_reviews[df_with_reviews["brand_type"] == bt]
    suspect  = eligible[eligible["is_rating_suspect"] == True]
    rate     = len(suspect) / len(eligible) * 100 if len(eligible) > 0 else 0
    print(f"  {bt:<20} {len(eligible):>8,} {len(suspect):>10,} {rate:>13.1f}%")
    summary_rows.append({
        "brand_type"    : bt,
        "eligible_count": len(eligible),
        "suspect_count" : len(suspect),
        "suspect_rate"  : round(rate, 2)
    })

# Tổng
total_eligible = len(df_with_reviews)
total_suspect  = df["is_rating_suspect"].sum()
total_rate     = total_suspect / total_eligible * 100 if total_eligible > 0 else 0
print(f"  {'TOTAL':<20} {total_eligible:>8,} {total_suspect:>10,} {total_rate:>13.1f}%")
print()

suspect_summary = pd.DataFrame(summary_rows)
print("✅ Cột 'is_rating_suspect' (bool) đã được thêm vào DataFrame.")
print()
print("── Sample sản phẩm suspect (OEM_Generic + Local_Generic):")
sample_suspects = df[
    (df["is_rating_suspect"]) &
    (df["brand_type"].isin(["OEM_Generic", "Local_Generic"]))
][["product_name", "brand_type", "brand_name", "price", "rating_average", "review_count"]].head(10)
print(sample_suspects.to_string(index=False))

RATING ANOMALY – SUSPECT RATE ANALYSIS
------------------------------------------------------------
  Ngưỡng: rating > 4.5 AND 1 <= review_count < 10

Tổng sản phẩm có review thực (review >= 1): 2,959
  (loại bỏ 8,019 sản phẩm new_listing / 0 review)

brand_type             eligible    suspect   suspect_rate
------------------------------------------------------------
  Global_Brand            1,401        791          56.5%
  Local_Generic           1,507        808          53.6%
  OEM_Generic                51         26          51.0%
  TOTAL                   2,959      1,625          54.9%

✅ Cột 'is_rating_suspect' (bool) đã được thêm vào DataFrame.

── Sample sản phẩm suspect (OEM_Generic + Local_Generic):
                                                                                            product_name    brand_type brand_name   price  rating_average  review_count
                                        Tai nghe (Headphones DJ) HDJ-CUE1 (Pioneer DJ) - Hàng Chính Hãng Loc

---
## Section 5 – Discount Flag: Phát hiện Giảm giá ảo

**4 nhãn `discount_flag`:**

| Flag | Điều kiện | Ý nghĩa |
|------|-----------|----------|
| `no_discount` | discount_rate == 0 | Không giảm giá |
| `fake_discount` | discount_rate > 0 AND price >= original_price | Giảm giá ảo: % cao nhưng giá bán ≥ giá gốc |
| `extreme_discount` | discount_rate >= 50 AND price < original_price | Flash sale cực cao (nguy cơ giá gốc bị thổi phồng) |
| `normal_discount` | Còn lại | Giảm giá hợp lệ thông thường |

In [8]:
print("DISCOUNT FLAG ANALYSIS")
print("-" * 60)

def assign_discount_flag(row):
    """
    Phân loại discount thành 4 nhóm để phát hiện chiến lược giảm giá ảo.

    Thứ tự ưu tiên kiểm tra:
    1. Không giảm giá
    2. Giảm giá ảo (discount_rate > 0 nhưng price >= original_price)
    3. Flash sale cực cao (discount >= EXTREME_THRESHOLD và giá thực sự giảm)
    4. Giảm giá bình thường
    """
    dr    = row["discount_rate"]
    price = row["price"]
    orig  = row["original_price"]

    if dr == 0:
        return "no_discount"
    if price >= orig and dr > 0:
        return "fake_discount"          # Ghi % giảm nhưng giá bán >= giá gốc
    if dr >= EXTREME_DISCOUNT_THRESHOLD and price < orig:
        return "extreme_discount"       # Giảm >= 50%, có thể giá gốc bị thổi
    return "normal_discount"

df["discount_flag"]    = df.apply(assign_discount_flag, axis=1)
df["is_fake_discount"] = (df["discount_flag"] == "fake_discount")

# ── Tổng quan discount flag ───────────────────────────────────────────────────
flag_counts = df["discount_flag"].value_counts()
print("\nPhân bố discount_flag (toàn bộ):")
for flag, cnt in flag_counts.items():
    pct = cnt / len(df) * 100
    print(f"  {flag:<20}: {cnt:>6,} ({pct:.1f}%)")
print()

# ── Phân tích fake_discount theo brand_type ───────────────────────────────────
print("Tỷ lệ 'fake_discount' theo brand_type:")
print(f"  {'brand_type':<20} {'total':>8} {'fake':>8} {'fake_%':>10} {'extreme':>10} {'extreme_%':>12}")
print("  " + "-" * 72)

discount_summary = []
for bt in sorted(df["brand_type"].unique()):
    sub   = df[df["brand_type"] == bt]
    total = len(sub)
    fake  = (sub["discount_flag"] == "fake_discount").sum()
    extr  = (sub["discount_flag"] == "extreme_discount").sum()
    fake_pct = fake / total * 100 if total > 0 else 0
    extr_pct = extr / total * 100 if total > 0 else 0
    print(f"  {bt:<20} {total:>8,} {fake:>8,} {fake_pct:>9.1f}% {extr:>10,} {extr_pct:>11.1f}%")
    discount_summary.append({
        "brand_type"         : bt,
        "total"              : total,
        "fake_discount_count": fake,
        "fake_discount_rate" : round(fake_pct, 2),
        "extreme_discount_count": extr,
        "extreme_discount_rate" : round(extr_pct, 2)
    })

print()
print("── Top 10 sản phẩm 'fake_discount' có discount_rate cao nhất:")
top_fake = df[
    df["discount_flag"] == "fake_discount"
][["product_name", "brand_type", "price", "original_price", "discount_rate"]]\
    .sort_values("discount_rate", ascending=False)\
    .head(10)
print(top_fake.to_string(index=False))

print(f"\n✅ Cột 'discount_flag' và 'is_fake_discount' đã được thêm vào DataFrame.")

DISCOUNT FLAG ANALYSIS
------------------------------------------------------------

Phân bố discount_flag (toàn bộ):
  no_discount         : 10,005 (91.1%)
  normal_discount     :    876 (8.0%)
  extreme_discount    :     97 (0.9%)

Tỷ lệ 'fake_discount' theo brand_type:
  brand_type              total     fake     fake_%    extreme    extreme_%
  ------------------------------------------------------------------------
  Global_Brand            4,514        0       0.0%         63         1.4%
  Local_Generic           6,307        0       0.0%         33         0.5%
  OEM_Generic               157        0       0.0%          1         0.6%

── Top 10 sản phẩm 'fake_discount' có discount_rate cao nhất:
Empty DataFrame
Columns: [product_name, brand_type, price, original_price, discount_rate]
Index: []

✅ Cột 'discount_flag' và 'is_fake_discount' đã được thêm vào DataFrame.


---
## Section 6 – Assertions: Kiểm tra tính toàn vẹn

In [9]:
print("ASSERTIONS – KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU")
print("=" * 60)

errors = []

# 1. Số dòng không thay đổi
if len(df) != ORIGINAL_ROW_COUNT:
    errors.append(f"❌ Row count changed: {ORIGINAL_ROW_COUNT} → {len(df)}")
else:
    print(f"  ✅ Row count bảo toàn: {len(df):,} dòng")

# 2. purchase_status chỉ có 2 giá trị
valid_statuses = {"has_sales", "new_listing"}
actual_statuses = set(df["purchase_status"].unique())
if not actual_statuses.issubset(valid_statuses):
    errors.append(f"❌ purchase_status có giá trị lạ: {actual_statuses - valid_statuses}")
else:
    print(f"  ✅ purchase_status: chỉ có {actual_statuses}")

# 3. discount_flag chỉ có 4 giá trị hợp lệ
valid_flags = {"no_discount", "fake_discount", "extreme_discount", "normal_discount"}
actual_flags = set(df["discount_flag"].unique())
if not actual_flags.issubset(valid_flags):
    errors.append(f"❌ discount_flag có giá trị lạ: {actual_flags - valid_flags}")
else:
    print(f"  ✅ discount_flag: {actual_flags} (tập hợp con hợp lệ)")

# 4. is_fake_discount = True phải có discount_rate > 0
bad_fake = df[(df["is_fake_discount"]) & (df["discount_rate"] == 0)]
if len(bad_fake) > 0:
    errors.append(f"❌ {len(bad_fake)} sản phẩm is_fake_discount=True nhưng discount_rate==0")
else:
    print(f"  ✅ Tất cả fake_discount đều có discount_rate > 0")

# 5. is_rating_suspect phải là boolean
if df["is_rating_suspect"].dtype != bool:
    errors.append(f"❌ is_rating_suspect không phải bool: {df['is_rating_suspect'].dtype}")
else:
    print(f"  ✅ is_rating_suspect: dtype = bool")

# 6. Không có product_id trùng
dup = df.duplicated(subset=["product_id"]).sum()
if dup > 0:
    errors.append(f"❌ Có {dup} product_id bị trùng lặp")
else:
    print(f"  ✅ Không có product_id trùng lặp")

# 7. Các cột mới phải tồn tại
new_cols = ["purchase_status", "is_rating_suspect", "discount_flag", "is_fake_discount"]
missing_cols = [c for c in new_cols if c not in df.columns]
if missing_cols:
    errors.append(f"❌ Thiếu cột: {missing_cols}")
else:
    print(f"  ✅ Tất cả cột mới tồn tại: {new_cols}")

print()
if errors:
    print("⚠️  CÓ LỖI PHÁT SINH:")
    for e in errors:
        print(f"  {e}")
    raise AssertionError("Preprocessing thất bại – kiểm tra các lỗi trên.")
else:
    print("✅ TẤT CẢ ASSERTIONS PASSED – Dữ liệu hoàn toàn toàn vẹn!")

ASSERTIONS – KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU
  ✅ Row count bảo toàn: 10,978 dòng
  ✅ purchase_status: chỉ có {'has_sales', 'new_listing'}
  ✅ discount_flag: {'normal_discount', 'no_discount', 'extreme_discount'} (tập hợp con hợp lệ)
  ✅ Tất cả fake_discount đều có discount_rate > 0
  ✅ is_rating_suspect: dtype = bool
  ✅ Không có product_id trùng lặp
  ✅ Tất cả cột mới tồn tại: ['purchase_status', 'is_rating_suspect', 'discount_flag', 'is_fake_discount']

✅ TẤT CẢ ASSERTIONS PASSED – Dữ liệu hoàn toàn toàn vẹn!


---
## Section 7 – Export: Lưu kết quả

In [12]:
import sqlite3
from datetime import datetime

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"[S7] Processed CSV: {OUTPUT_CSV}")
print(f"     {len(df):,} rows | {len(df.columns)} columns")

# Export to SQLite
with sqlite3.connect(OUTPUT_DB) as conn:
    df.to_sql("products", conn, if_exists="replace", index=False)
print(f"[S7] Processed DB: {OUTPUT_DB}")

# ── Helper: chuyển numpy types → native Python (tránh JSON serialize lỗi) ────
def _to_native(obj):
    """Recursively convert numpy int/float to Python int/float for JSON."""
    if isinstance(obj, dict):
        return {k: _to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

report = {
    "metadata": {"processed_at": datetime.now().isoformat(),
                 "input_csv": INPUT_CSV, "output_csv": OUTPUT_CSV,
                 "output_db": OUTPUT_DB,
                 "total_rows": len(df), "total_columns": len(df.columns)},
    "dead_columns_dropped": dead_columns,
    "purchase_status": {s: int(c) for s, c in df["purchase_status"].value_counts().items()},
    "purchase_status_pct": {s: round(float(c/len(df)*100), 2) for s, c in df["purchase_status"].value_counts().items()},
    "rating_anomaly": {
        "thresholds": {"rating_above": SUSPECT_RATING_THRESHOLD, "review_min": SUSPECT_REVIEW_MIN, "review_max": SUSPECT_REVIEW_MAX},
        "by_brand_type": {r["brand_type"]: {"eligible_count": int(r["eligible_count"]), "suspect_count": int(r["suspect_count"]), "suspect_rate_pct": float(r["suspect_rate"])} for r in suspect_summary.to_dict("records")}
    },
    "discount_analysis": {
        "thresholds": {"extreme_discount_pct": EXTREME_DISCOUNT_THRESHOLD},
        "distribution": {f: int(c) for f, c in df["discount_flag"].value_counts().items()},
        "by_brand_type": _to_native({r["brand_type"]: r for r in discount_summary})
    }
}
with open(REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"[S7] Report JSON: {REPORT_JSON}")
print()
print("PIPELINE VERIFICATION COMPLETE - TAT CA SECTIONS PASSED!")
print(f"Cot moi: {new_cols}")
print(f"Cot bi drop: {dead_columns if dead_columns else 'Khong co'}")

[S7] Processed CSV: c:\Users\Admin\Documents\GitHub\Ecom-Data-Crawler\data/tiki_electronics_2026_processed.csv
     10,978 rows | 20 columns
[S7] Processed DB: c:\Users\Admin\Documents\GitHub\Ecom-Data-Crawler\data/tiki_electronics_2026_processed.db
[S7] Report JSON: c:\Users\Admin\Documents\GitHub\Ecom-Data-Crawler\data\preprocessing_report.json

PIPELINE VERIFICATION COMPLETE - TAT CA SECTIONS PASSED!
Cot moi: ['purchase_status', 'is_rating_suspect', 'discount_flag', 'is_fake_discount']
Cot bi drop: Khong co
